# Strands Agent with Datadog APM Observability on Amazon Bedrock AgentCore Runtime

## Overview

This notebook demonstrates deploying a Strands agent to Amazon Bedrock AgentCore Runtime with Datadog APM observability. The implementation uses Strands' built-in `StrandsTelemetry` to automatically instrument agent traces and export them to Datadog APM through OpenTelemetry (OTEL) — the simplest path to observability with zero custom OTEL code.

> **Looking for GenAI-specific observability?** For prompt/completion tracking, token usage analysis, and evaluations, see the companion [LLM Observability tutorial](../runtime_with_strands_and_datadog.ipynb) which uses custom OpenTelemetry instrumentation to route traces to Datadog LLM Observability.

## Key Components

- **Strands Agents**: Python framework for building LLM-powered agents with built-in telemetry support
- **Amazon Bedrock AgentCore Runtime**: Managed runtime service for hosting and scaling agents on AWS
- **Datadog APM**: Application Performance Monitoring with distributed trace views
- **OpenTelemetry**: Industry-standard protocol for collecting and exporting telemetry data

## Architecture

The agent is containerized and deployed to AgentCore Runtime, which provides HTTP endpoints for invocation. Telemetry data flows from the Strands agent through an OTLP exporter directly to Datadog's trace endpoint for monitoring and debugging. OTEL configuration is passed as environment variables at deployment time, keeping the agent code clean and portable.

## Prerequisites

- Python 3.10+
- AWS credentials configured with Bedrock and AgentCore permissions
- [Datadog](https://www.datadoghq.com/) account with API key
- Access to Amazon Bedrock Claude models in your configured region

In [ ]:
!pip install --force-reinstall -U -r requirements.txt

## Configure AWS Credentials

## Agent Implementation

The agent file (`strands_claude.py`) implements a travel assistant with web search. Key points:

- **Module-level telemetry init**: `StrandsTelemetry` sets up a global tracer at import time
- **`DD_API_KEY` read at runtime**: The agent checks for the key and conditionally enables telemetry
- **OTLP configuration via env vars**: OTEL exporter settings are passed as environment variables at launch time, not hardcoded in the agent
- **Automatic trace export**: All agent invocations, tool calls, and LLM interactions are automatically traced and sent to Datadog

In [ ]:
%%writefile strands_claude.py
import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from ddgs import DDGS

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())

# Datadog OTLP configuration is passed as env vars at launch time.
# StrandsTelemetry reads these to configure the exporter.
dd_api_key = os.getenv("DD_API_KEY")
if dd_api_key:
    # Initialize StrandsTelemetry at module level (sets up global tracer)
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()
    logger.info("\u2713 Datadog telemetry initialized with OTLP exporter")
else:
    logger.warning("\u26a0 No DD_API_KEY provided, running without Datadog telemetry")


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )

        return "\n".join(formatted_results) if formatted_results else "No results found."

    except Exception as e:
        return f"Error searching the web: {str(e)}"


def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")

    bedrock_model = BedrockModel(
        model_id=model_id,
        region_name=region,
        temperature=0.0,
        max_tokens=1024
    )
    return bedrock_model


bedrock_model = get_bedrock_model()

system_prompt = """You are an experienced travel agent specializing in personalized travel recommendations
with access to real-time web information. Your role is to find dream destinations matching user preferences
using web search for current information. You should provide comprehensive recommendations with current
information, brief descriptions, and practical travel details."""

app = BedrockAgentCoreApp()


def initialize_agent():
    """Initialize the agent (telemetry is already configured at module level)."""
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[web_search]
    )
    return agent


@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    logger.info("[%s] User input: %s", context.session_id, user_input)

    agent = initialize_agent()

    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()

### Configure AgentCore Runtime deployment

Next we will use the starter toolkit to configure the AgentCore Runtime deployment with an entrypoint and a requirements file. The toolkit will auto-create the execution role and Amazon ECR repository on launch.

During the configure step, your Dockerfile will be generated based on your application code. When using the `bedrock_agentcore_starter_toolkit` to configure your agent, it enables AgentCore Observability by default — to use Datadog instead, set `disable_otel=True` to disable AgentCore's built-in ADOT pipeline.

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_datadog_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode="NO_MEMORY",
    disable_otel=True,
)
response

## Deploy to AgentCore Runtime

Now that we've got a Dockerfile, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

### Datadog Configuration

OTLP configuration is passed as environment variables at launch time. `StrandsTelemetry` reads these to configure the exporter — no OTEL settings are hardcoded in the agent code.

To send traces to Datadog, you need:
- **Datadog API Key**: Get this from your Datadog account at Organization Settings → API Keys

The following OTEL env vars are set in the `launch()` call:
- **`OTEL_EXPORTER_OTLP_TRACES_ENDPOINT`**: Datadog's OTLP trace intake endpoint
- **`OTEL_EXPORTER_OTLP_TRACES_HEADERS`**: API key and `dd-otlp-source=datadog` for APM routing
- **`OTEL_EXPORTER_OTLP_TRACES_PROTOCOL`**: `http/protobuf`
- **`OTEL_SEMCONV_STABILITY_OPT_IN`**: `gen_ai_latest_experimental` for GenAI semantic conventions

**Datadog region endpoints:**
- US1: `https://trace.agent.datadoghq.com/v1/traces` (default)
- US3: `https://trace.agent.us3.datadoghq.com/v1/traces`
- US5: `https://trace.agent.us5.datadoghq.com/v1/traces`
- EU1: `https://trace.agent.datadoghq.eu/v1/traces`
- AP1: `https://trace.agent.ap1.datadoghq.com/v1/traces`

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [ ]:
# Datadog configuration
datadog_api_key = "<your-datadog-api-key>"  # Replace with your Datadog API key

launch_result = agentcore_runtime.launch(
    env_vars={
        "DD_API_KEY": datadog_api_key,
        "OTEL_EXPORTER_OTLP_TRACES_PROTOCOL": "http/protobuf",
        "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT": "https://trace.agent.datadoghq.com/v1/traces",
        "OTEL_EXPORTER_OTLP_TRACES_HEADERS": f"dd-api-key={datadog_api_key},dd-otlp-source=datadog",
        "OTEL_SEMCONV_STABILITY_OPT_IN": "gen_ai_latest_experimental",
        "DISABLE_ADOT_OBSERVABILITY": "true",
    }
)
launch_result

## Check Deployment Status

Wait for the runtime to be ready before invoking:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="../images/invoke.png" width="75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What are the best beach destinations in Southeast Asia for a week-long trip in December?"})

In [ ]:
from IPython.display import Markdown, display
display(Markdown("".join(invoke_response['response'])))

## View Traces in Datadog APM

After invoking the agent, traces appear in Datadog within a few minutes:

1. Go to [Datadog APM → Traces](https://app.datadoghq.com/apm/traces)
2. Filter by service: `strands-agents` (Strands' default service name)

The traces include agent invocations, tool calls, and model interactions:

<div style="text-align:left">
    <img src="apm-trace-example.png" width="75%"/>
</div>

### Datadog APM Features

Datadog APM provides infrastructure-level observability for your agent:

- **Trace Explorer**: View end-to-end agent traces with tool calls and model interactions in a single timeline
- **Service Map**: Visualize dependencies between your agent, tools, and model providers
- **Flame Graphs**: Drill into individual traces to see exact timing of each operation
- **Latency Analysis**: Track response times for model invocations and tool calls
- **Error Monitoring**: Identify failed model calls, tool errors, and agent exceptions with full context

> **Want GenAI-specific views?** For prompt/completion content tracking, token usage analysis, and evaluations, see the companion [LLM Observability tutorial](../runtime_with_strands_and_datadog.ipynb) which routes traces to [Datadog LLM Observability](https://docs.datadoghq.com/llm_observability/) instead of APM.

For more information, see the [Datadog APM documentation](https://docs.datadoghq.com/tracing/).

## Cleanup (Optional)

Clean up the deployed resources:

In [ ]:
import boto3

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

## Summary

You have successfully deployed a Strands agent to Amazon Bedrock AgentCore Runtime with Datadog APM observability. The implementation demonstrates:

- Using `StrandsTelemetry` for automatic OTEL trace instrumentation — zero custom OpenTelemetry code required
- Passing OTLP configuration as deployment-level environment variables (not hardcoded in agent code)
- Using `dd-otlp-source=datadog` to route traces to Datadog APM
- Enabling GenAI semantic conventions with `OTEL_SEMCONV_STABILITY_OPT_IN`
- Disabling AgentCore's built-in ADOT to use Datadog instead

This tutorial uses the simplest instrumentation path — Strands' built-in `StrandsTelemetry` — to send traces to Datadog APM for general-purpose application monitoring. For GenAI-specific observability including prompt/completion tracking, token usage analysis, and evaluations, see the companion [LLM Observability tutorial](../runtime_with_strands_and_datadog.ipynb).

### Resources

- [AgentCore Observability docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html)
- [Datadog APM documentation](https://docs.datadoghq.com/tracing/)
- [Datadog LLM Observability — OTEL Instrumentation](https://docs.datadoghq.com/llm_observability/instrumentation/otel_instrumentation/)
- [Strands Agents Observability](https://strandsagents.com/latest/documentation/docs/user-guide/observability-evaluation/observability/)
- [OpenTelemetry GenAI Semantic Conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/)